In [1]:
import pandas as pd
from pathlib import Path


In [2]:
products_path = Path("../data/processed/products_features.csv")

if not products_path.exists():
    raise FileNotFoundError("Exécute d'abord le notebook 03_feature_engineering.ipynb.")

products = pd.read_csv(products_path, encoding="utf-8-sig")
products["product_id"] = products["product_id"].astype(str)

print("Produits disponibles :", len(products))
display(products.head())


Produits disponibles : 9


,product_id,name,domain,category,subcategory,brand,model,price_dzd,availability,feature_text,...,pressure_rating_bar,power_kw,voltage_v,price_dzd.1,diameter_mm_missing,length_m_missing,pressure_rating_bar_missing,power_kw_missing,voltage_v_missing,price_dzd_missing
0,SH001,chauffe-bain 11l junkers,chauffage,chaudière / générateur,chauffe-bain gaz,junkers,NaN,28500.0,In stock,chauffe-bain 11l junkers chauffage chaudière /...,...,NaN,NaN,NaN,28500.0,1,1,1,1,1,0
1,SH002,chauffe-eau smart 12l energical,chauffage,chaudière / générateur,chauffe-eau gaz,energical,smart 12l,29500.0,In stock,chauffe-eau smart 12l energical chauffage chau...,...,NaN,NaN,NaN,29500.0,1,1,1,1,1,0
2,SH003,chaudière ventouse 28 kw energical,chauffage,chaudière / générateur,chaudière gaz,energical,28 kw ventouse,NaN,NaN,chaudière ventouse 28 kw energical chauffage c...,...,NaN,28.0,NaN,NaN,1,1,1,0,1,1
3,SH004,accélérateur grundfos 15-50 75w réf. 99062433,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-50,21800.0,NaN,accélérateur grundfos 15-50 75w réf. 99062433 ...,...,NaN,NaN,NaN,21800.0,1,1,1,1,1,0
4,SH005,accélérateur grundfos 15-60 110w réf. 99157090,chauffage,pompe de circulation,circulateur / accélérateur chaudière,grundfos,15-60,21300.0,NaN,accélérateur grundfos 15-60 110w réf. 99157090...,...,NaN,NaN,NaN,21300.0,1,1,1,1,1,0


In [11]:


compatibility_path = Path("../dataraw/compatibility.csv")

if not compatibility_path.exists():
    raise FileNotFoundError(
        "Crée data/raw/compatibility.csv avec les colonnes du modèle fourni."
    )

relations = pd.read_csv(compatibility_path, encoding="utf-8-sig")

required_relation_columns = [
    "product_id", "related_product_id", "relation_type",
    "conditions", "evidence_source_id", "verification_status"
]
missing = [c for c in required_relation_columns if c not in relations.columns]
if missing:
    raise ValueError(f"Colonnes manquantes dans compatibility.csv : {missing}")

for col in ["product_id", "related_product_id"]:
    relations[col] = relations[col].astype("string").str.strip()

print("Relations enregistrées :", len(relations))
display(relations.head())



Relations enregistrées : 0


,relation_id,product_id,related_product_id,relation_type,conditions,evidence_source_id,verified_by,verification_status,notes


In [12]:
known_ids = set(products["product_id"].dropna().astype(str))

relations["source_exists"] = relations["product_id"].isin(known_ids)
relations["target_exists"] = relations["related_product_id"].isin(known_ids)

invalid_relations = relations[
    (~relations["source_exists"]) | (~relations["target_exists"])
]

print("Relations avec identifiants inconnus :", len(invalid_relations))
display(invalid_relations)


Relations avec identifiants inconnus : 0


,relation_id,product_id,related_product_id,relation_type,conditions,evidence_source_id,verified_by,verification_status,notes,source_exists,target_exists


In [13]:
verified = relations[
    relations["verification_status"].astype("string").str.strip().str.lower().eq("verified")
    & relations["source_exists"]
    & relations["target_exists"]
    & relations["evidence_source_id"].notna()
    & relations["evidence_source_id"].astype("string").str.strip().ne("")
].copy()

print("Relations vérifiées et documentées :", len(verified))
display(verified)


Relations vérifiées et documentées : 0


,relation_id,product_id,related_product_id,relation_type,conditions,evidence_source_id,verified_by,verification_status,notes,source_exists,target_exists


In [14]:
allowed_relation_types = {
    "compatible_with",
    "complementary_to",
    "alternative_to"
}

verified["relation_type"] = (
    verified["relation_type"].astype("string").str.strip().str.lower()
)

verified = verified[verified["relation_type"].isin(allowed_relation_types)].copy()

print("Relations retenues par type :")
display(verified["relation_type"].value_counts())


Relations retenues par type :


Series([], Name: count, dtype: Int64)

In [15]:
product_info = products[
    [c for c in ["product_id", "name", "domain", "category", "brand", "model", "price_dzd"]
     if c in products.columns]
].copy()

source_info = product_info.add_prefix("source_")
target_info = product_info.add_prefix("target_")

enriched = verified.merge(
    source_info, left_on="product_id", right_on="source_product_id", how="left"
)
enriched = enriched.merge(
    target_info, left_on="related_product_id", right_on="target_product_id", how="left"
)

display(enriched.head())


,relation_id,product_id,related_product_id,relation_type,conditions,evidence_source_id,verified_by,verification_status,notes,source_exists,...,source_brand,source_model,source_price_dzd,target_product_id,target_name,target_domain,target_category,target_brand,target_model,target_price_dzd


In [16]:
def recommend_by_rules(product_id, relation_types=None):
    product_id = str(product_id).strip()

    candidates = enriched[enriched["product_id"].astype(str) == product_id].copy()

    if relation_types is not None:
        if isinstance(relation_types, str):
            relation_types = [relation_types]
        candidates = candidates[candidates["relation_type"].isin(relation_types)]

    result_columns = [
        c for c in [
            "related_product_id", "target_name", "target_domain",
            "target_category", "target_brand", "target_model",
            "relation_type", "conditions", "evidence_source_id"
        ] if c in candidates.columns
    ]

    return candidates[result_columns].reset_index(drop=True)


In [17]:
if products.empty:
    print("Le catalogue ne contient aucun produit.")
else:
    selected_id = products.iloc[0]["product_id"]
    print("Produit sélectionné :", selected_id)
    display(recommend_by_rules(selected_id))


Produit sélectionné : SH001


,related_product_id,target_name,target_domain,target_category,target_brand,target_model,relation_type,conditions,evidence_source_id


In [18]:
if products.empty:
    print("Le catalogue ne contient aucun produit.")
else:
    selected_id = products.iloc[0]["product_id"]
    print("Produit sélectionné :", selected_id)
    display(recommend_by_rules(selected_id))


Produit sélectionné : SH001


,related_product_id,target_name,target_domain,target_category,target_brand,target_model,relation_type,conditions,evidence_source_id


In [19]:
not_verified = relations[
    ~relations.index.isin(verified.index)
].copy()

print("Relations non retenues :", len(not_verified))
display(not_verified)


Relations non retenues : 0


,relation_id,product_id,related_product_id,relation_type,conditions,evidence_source_id,verified_by,verification_status,notes,source_exists,target_exists


In [20]:
output_path = Path("../data/processed/knowledge_recommendations.csv")
output_path.parent.mkdir(parents=True, exist_ok=True)

export_columns = [
    c for c in [
        "product_id", "related_product_id", "relation_type",
        "conditions", "evidence_source_id", "verification_status",
        "source_name", "target_name", "target_domain", "target_category",
        "target_brand", "target_model"
    ] if c in enriched.columns
]

knowledge_recommendations = enriched[export_columns].copy()
knowledge_recommendations.to_csv(
    output_path, index=False, encoding="utf-8-sig"
)

print("Fichier créé :", output_path.exists())
print("Nombre de recommandations :", len(knowledge_recommendations))


Fichier créé : True
Nombre de recommandations : 0
